# Step 1: Generate Weight Configurations via K-means

## Overview
This notebook generates a set of representative weight vectors for the multi-objective optimization. It starts by creating an exhaustive set of all possible integer weight combinations and then uses the K-means algorithm to sub-sample a smaller, computationally feasible set of points that evenly cover the objective space.

In [5]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
import os

In [6]:
# --- Parameters to modify ---

# The number of objectives for the optimization
N_OBJECTIVES = 3

# The total sum that the weights of each combination must have
TOTAL_SUM = 20

# The final number of representative weight sets to obtain
N_CLUSTERS = 16

# --- Output file ---
OUTPUT_DIR = "../results"
OUTPUT_FILENAME = os.path.join(OUTPUT_DIR, "01_weights.csv")

# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [7]:
def generate_combinations(n_objectives, total_sum):
    """Recursively generates all non-negative integer combinations."""
    if n_objectives < 1: return []
    if n_objectives == 1: return [[total_sum]]
    combinations = []
    for i in range(total_sum + 1):
        for sub_combo in generate_combinations(n_objectives - 1, total_sum - i):
            combinations.append([i] + sub_combo)
    return combinations

def find_representative_points(full_set, n_clusters):
    """Performs K-means clustering to find representative points."""
    print(f"Applying K-means with n_clusters={n_clusters}...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
    kmeans.fit(full_set)

    distances = cdist(kmeans.cluster_centers_, full_set)
    closest_point_indices = np.argmin(distances, axis=1)
    unique_indices = np.unique(closest_point_indices)

    print(f"Found {len(unique_indices)} unique representative points.")
    return full_set[unique_indices]

In [8]:
# 1. Generate the Full Set of Combinations
full_set = np.array(generate_combinations(N_OBJECTIVES, TOTAL_SUM))
print(f"Full space generated: {full_set.shape[0]} total combinations.")

# 2. Run K-means Sub-sampling
representative_points = find_representative_points(full_set, N_CLUSTERS)

# 3. Save Results
headers = [f"obj_{i+1}" for i in range(N_OBJECTIVES)]
df_weights = pd.DataFrame(representative_points, columns=headers)
df_weights_sorted = df_weights.sort_values(by=headers).reset_index(drop=True)

df_weights_sorted.to_csv(OUTPUT_FILENAME, index=False)
print(f"\n✅ Weight sets saved to '{OUTPUT_FILENAME}'")

print("\nPreview of the generated weights:")
display(df_weights_sorted.head())

Full space generated: 231 total combinations.
Applying K-means with n_clusters=16...
Found 16 unique representative points.

✅ Weight sets saved to '../results/01_weights.csv'

Preview of the generated weights:


,obj_1,obj_2,obj_3
0,1,1,18
1,1,10,9
2,1,13,6
3,1,17,2
4,2,5,13
